In [ ]:
import logging

from sqlglot import expressions as exp

from CCPipeline import CCPipeline
from config.config import load_config
from database.TablePartitioner import TablePartitioner
from dataset import label_list, sql_list
from model.SQLOptimizer import SQLOptimizer

logging.basicConfig(
    filename="output.log",  # 指定文件名
    level=logging.INFO,  # 记录 debug 以上的所有日志
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)


config = load_config()
partitioner = TablePartitioner(config)

metadata = partitioner.get_schema_metadata()

# # 加载配置
from sql_for_test_1 import example_sql

so = SQLOptimizer(example_sql, metadata)

left_1 = so.alias_dict[so.join_expr[0].this.table]
right_1 = so.alias_dict[so.join_expr[0].expression.table]
table_map1 = [
    {left_1: left, right_1: right}
    for left in metadata[left_1]["part_table"]
    for right in metadata[right_1]["part_table"]
]

sql_list = []
label_list = []
for table_map in table_map1:
    parsed_query_copy = so.parsed_query.copy()
    for table in parsed_query_copy.find_all(exp.Table):
        new_table = exp.Table(
            this=exp.to_identifier(table_map[table.name]),
            db=exp.to_identifier(table.name),
            alias=table.args.get("alias"),
        )
        table.replace(new_table)
    new_sql = parsed_query_copy.sql()
    sql_list.append(new_sql)
    result = partitioner.execute_query(new_sql).fetchone()
    if result is None:
        label_list.append(0)
    else:
        label_list.append(result[0])

logging.info("finish generating sql list and label list")



In [ ]:
import logging

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split

from model.BloomFilter import BloomFilter
from model.CardinalityClassifier import CardinalityClassifier
from model.SQLDataLoader import SQLDataLoader
from model.SQLDataset import SQLDataset
from model.SQLOptimizer import SQLOptimizer
from database.TablePartitioner import TablePartitioner

logging.basicConfig(
    filename="output.log",  # 指定文件名
    level=logging.INFO,  # 记录 debug 以上的所有日志
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)


class CCPipeline:
    def __init__(
        self, partitioner: TablePartitioner, sql_list, label_list, config=None
    ):
        self.partitioner = partitioner
        self.sql_list = sql_list
        self.label_list = [1 if label > 0 else 0 for label in label_list]
        self.config = config or {
            "batch_size": 32,
            "epochs": 20,
            "lr": 1e-3,
            "threshold": 0.5,
            "use_oversample": True,
        }
        self.model = CardinalityClassifier(partitioner)
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.config["lr"])

    def _build_dataset(self):
        dataset = SQLDataset(self.model, self.sql_list, self.label_list)
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(
            dataset, [train_size, val_size]
        )

        self.train_loader = SQLDataLoader(
            self.train_dataset,
            batch_size=self.config["batch_size"],
            use_oversample=self.config["use_oversample"],
        )
        self.val_loader = SQLDataLoader(
            self.val_dataset, batch_size=self.config["batch_size"], shuffle=False
        )
        logging.info(
            f"Train Size: {len(self.train_dataset)}, Val Size: {len(self.val_dataset)}"
        )

    def train(self):
        self._build_dataset()
        for epoch in range(self.config["epochs"]):
            self.model.train()
            total_loss, total_correct, total_samples = 0, 0, 0
            for table_feats, join_feats, predicate_feats, labels in self.train_loader:
                self.optimizer.zero_grad()
                outputs = self.model(table_feats, join_feats, predicate_feats)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()
                preds = (torch.sigmoid(outputs) > 0.5).float()
                total_correct += (preds == labels).sum().item()
                total_samples += labels.size(0)

            train_acc = total_correct / total_samples
            train_loss = total_loss / len(self.train_loader)

            val_acc, val_loss = self._evaluate()
            logging.info(
                f"[Epoch {epoch}] Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}"
            )

    def _evaluate(self):
        self.model.eval()
        total_loss, total_correct, total_samples = 0, 0, 0
        with torch.no_grad():
            for table_feats, join_feats, predicate_feats, labels in self.val_loader:
                outputs = self.model(table_feats, join_feats, predicate_feats)
                loss = self.criterion(outputs, labels)
                preds = (torch.sigmoid(outputs) > 0.5).float()
                total_loss += loss.item()
                total_correct += (preds == labels).sum().item()
                total_samples += labels.size(0)
        return total_correct / total_samples, total_loss / len(self.val_loader)

    # def get_all_results(self)

    def predict_by_pit(self, so: SQLOptimizer):
        ltable, lcol, rtable, rcol = so.get_condition_from_expr(so.join_expr[0])
        lhs_pit = so.get_partition_index_tree(ltable, lcol)
        rhs_interval = so.select_intervals(rtable, rcol)
        ret = []
        for interval in rhs_interval:
            ret.extend(lhs_pit.query_interval_overlap(interval))
        return ret

    def predict(
        self, so: SQLOptimizer, sql_list: list[str]
    ) -> tuple[list[bool], list[float]]:
        """
        封装完整推理流程：先 cc 预测，再 PIT，再 Bloom
        """
        results = []
        self.model.eval()
        with torch.no_grad():
            so_list = [
                SQLOptimizer(s, self.partitioner.get_schema_metadata())
                for s in sql_list
            ]
            table_feats, join_feats, pred_feats, pred_mask = self.model.build_batch(
                so_list
            )
            probs = torch.sigmoid(
                self.model(table_feats, join_feats, pred_feats, pred_mask)
            ).view(-1)
            logger.info(f"Predictions: {probs}")
        for idx, prob in enumerate(probs):
            if prob.item() > self.config["threshold"]:
                results.append(True)
                continue  # classifier 已判定为可行
            so = so_list[idx]
            ltable, lcol, rtable, rcol = so.get_condition_from_expr(so.join_expr[0])
            lschema = ltable.split("_")[0]
            rschema = rtable.split("_")[0]
            # === 尝试 PIT 判断 ===
            try:
                print(f"Try to use PIT for {so.query}, idx:{idx}")
                if self._check_pit(so, lschema, lcol, rschema, rtable):
                    results.append(True)
                    continue
            except Exception:
                pass

            # === 最后尝试 Bloom 判断 ===
            try:
                print(f"Try to use Bloom for {so.query}, idx:{idx}")
                # for test
                # results.append(False)
                results.append(self._check_bloom(lschema, ltable, lcol, rschema, rtable, rcol))
            except Exception:
                results.append(False)

        return results, probs.tolist()

    def _check_pit(self, so: SQLOptimizer, ltable, lcol, rtable, rcol):
        try:
            lhs_pit = so.get_partition_index_tree(ltable, lcol)
            rhs_interval = so.select_intervals(rtable, rcol)[0]
            return lhs_pit.query_interval_overlap(rhs_interval)
        except Exception as e:
            print(e)
            return False

    def _check_bloom(self, lschema, ltable,lcol, rschema, rtable,rcol):
        try:
            rdata = self.partitioner.select_data(rschema, rtable, [rcol])
            bf = BloomFilter(max(1, len(rdata)), 0.01)
            for row in rdata:
                if row:
                    bf.add(str(row[0]))
            ldata = self.partitioner.select_data(lschema, ltable, [lcol])
            for row in ldata:
                if str(row[0]) in bf:
                    return True
        except Exception:
            pass
        return False


In [52]:
ccp = CCPipeline(partitioner, sql_list, label_list)
ccp.train()
logging.info("finish loading data & training (if any)")

In [9]:
so = SQLOptimizer(example_sql, metadata)
res = ccp.predict(so, sql_list)

Try to use PIT for SELECT COUNT(*) FROM partsupp.partsupp_0 AS ps, lineitem.lineitem_1 AS l WHERE ps.ps_partkey = l.l_partkey AND ps.ps_availqty > 2000, idx:1
'lineitem_1_start'
Try to use Bloom for SELECT COUNT(*) FROM partsupp.partsupp_0 AS ps, lineitem.lineitem_1 AS l WHERE ps.ps_partkey = l.l_partkey AND ps.ps_availqty > 2000, idx:1
Try to use PIT for SELECT COUNT(*) FROM partsupp.partsupp_0 AS ps, lineitem.lineitem_2 AS l WHERE ps.ps_partkey = l.l_partkey AND ps.ps_availqty > 2000, idx:2
'lineitem_2_start'
Try to use Bloom for SELECT COUNT(*) FROM partsupp.partsupp_0 AS ps, lineitem.lineitem_2 AS l WHERE ps.ps_partkey = l.l_partkey AND ps.ps_availqty > 2000, idx:2
Try to use PIT for SELECT COUNT(*) FROM partsupp.partsupp_0 AS ps, lineitem.lineitem_3 AS l WHERE ps.ps_partkey = l.l_partkey AND ps.ps_availqty > 2000, idx:3
'lineitem_3_start'
Try to use Bloom for SELECT COUNT(*) FROM partsupp.partsupp_0 AS ps, lineitem.lineitem_3 AS l WHERE ps.ps_partkey = l.l_partkey AND ps.ps_availq

In [24]:
so = SQLOptimizer(example_sql, metadata)
ltable, lcol, rtable, rcol = so.get_condition_from_expr(so.join_expr[0])
lhs_pit = so.get_partition_index_tree(ltable, lcol)
rhs_interval = so.select_intervals(rtable, rcol)
ret = []
for interval in rhs_interval:
    ret.extend(lhs_pit.query_interval_overlap(interval))
ret


[('partsupp_1', 'lineitem_0'),
 ('partsupp_0', 'lineitem_0'),
 ('partsupp_1', 'lineitem_1'),
 ('partsupp_2', 'lineitem_1'),
 ('partsupp_3', 'lineitem_2'),
 ('partsupp_2', 'lineitem_2'),
 ('partsupp_3', 'lineitem_3'),
 ('partsupp_5', 'lineitem_3'),
 ('partsupp_4', 'lineitem_3'),
 ('partsupp_5', 'lineitem_4'),
 ('partsupp_6', 'lineitem_4'),
 ('partsupp_7', 'lineitem_5'),
 ('partsupp_6', 'lineitem_5'),
 ('partsupp_7', 'lineitem_6'),
 ('partsupp_8', 'lineitem_6'),
 ('partsupp_9', 'lineitem_7'),
 ('partsupp_10', 'lineitem_7'),
 ('partsupp_11', 'lineitem_8'),
 ('partsupp_10', 'lineitem_8'),
 ('partsupp_11', 'lineitem_9'),
 ('partsupp_12', 'lineitem_9'),
 ('partsupp_13', 'lineitem_10'),
 ('partsupp_12', 'lineitem_10'),
 ('partsupp_14', 'lineitem_10'),
 ('partsupp_15', 'lineitem_11'),
 ('partsupp_14', 'lineitem_11'),
 ('partsupp_15', 'lineitem_12'),
 ('partsupp_16', 'lineitem_12'),
 ('partsupp_17', 'lineitem_13'),
 ('partsupp_16', 'lineitem_13'),
 ('partsupp_19', 'lineitem_14'),
 ('partsupp_18

In [11]:
label_check = [1 if label > 0 else 0 for label in label_list]
for idx, value in enumerate(res[0]):
    if label_check[idx] != value:
        print(f"{idx},Error: {label_check[idx]} != {value}")

71,Error: 1 != False
285,Error: 1 != False
332,Error: 1 != False
356,Error: 1 != False
357,Error: 1 != False
452,Error: 1 != False
500,Error: 1 != False
547,Error: 1 != False
594,Error: 1 != False
595,Error: 1 != False
642,Error: 1 != False
666,Error: 1 != False


In [ ]:
error_sql = sql_list[71]
error_so = SQLOptimizer(error_sql, metadata)

ltable, lcol, rtable, rcol = error_so.get_condition_from_expr(error_so.join_expr[0])
lschema = ltable.split("_")[0]
rschema = rtable.split("_")[0]
rdata = ccp.partitioner.select_data(rschema, rtable, [rcol])
bf = BloomFilter(max(1, len(rdata)), 0.01)
for row in rdata:
    if row:
        bf.add(str(row[0]))
ldata = ccp.partitioner.select_data(lschema, ltable, [lcol])
# for row in ldata:
#     if row and row[0] in bf:
#         return True

In [48]:
for row in ldata:
    if str(row[0]) in bf:
        print("OK")
    break
ldata

OK


[(210000,),
 (210000,),
 (210000,),
 (210000,),
 (210001,),
 (210001,),
 (210001,),
 (210001,),
 (210002,),
 (210002,),
 (210002,),
 (210002,),
 (210003,),
 (210003,),
 (210003,),
 (210003,),
 (210004,),
 (210004,),
 (210004,),
 (210004,),
 (210005,),
 (210005,),
 (210005,),
 (210005,),
 (210006,),
 (210006,),
 (210006,),
 (210006,),
 (210007,),
 (210007,),
 (210007,),
 (210007,),
 (210008,),
 (210008,),
 (210008,),
 (210008,),
 (210009,),
 (210009,),
 (210009,),
 (210009,),
 (210010,),
 (210010,),
 (210010,),
 (210010,),
 (210011,),
 (210011,),
 (210011,),
 (210011,),
 (210012,),
 (210012,),
 (210012,),
 (210012,),
 (210013,),
 (210013,),
 (210013,),
 (210013,),
 (210014,),
 (210014,),
 (210014,),
 (210014,),
 (210015,),
 (210015,),
 (210015,),
 (210015,),
 (210016,),
 (210016,),
 (210016,),
 (210016,),
 (210017,),
 (210017,),
 (210017,),
 (210017,),
 (210018,),
 (210018,),
 (210018,),
 (210018,),
 (210019,),
 (210019,),
 (210019,),
 (210019,),
 (210020,),
 (210020,),
 (210020,),
 (21